<a href="https://colab.research.google.com/github/joohyosung/KMU_AIstudy/blob/main/week06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q stable-baselines3 "gymnasium[classic-control]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 80.4 MB/s eta 0:00:00


In [2]:
import gymnasium as gym  # 강화학습 환경 라이브러리


In [3]:
env = gym.make("CartPole-v1")  # 카트 위 막대 세우기 환경 생성


In [4]:
env.observation_space  # 관찰 공간


Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

In [5]:
env.action_space  # 행동 공간


Discrete(2)

In [8]:
import random
import numpy as np

# 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 환경 초기화 (시드 설정)
observation, info = env.reset(seed=SEED)
env.action_space.seed(SEED)

steps = 0
done = False

while not done:
    # 무작위 행동 선택
    action = env.action_space.sample()

    # 환경 한 스텝 진행
    observation, reward, terminated, truncated, info = env.step(action)
    steps += 1

    # 에피소드 종료 조건 확인 (종료 또는 시간 초과)
    if terminated or truncated:
        done = True

print(f"에피소드 종료 - 버틴 총 스텝 수: {steps}")

에피소드 종료 - 버틴 총 스텝 수: 30


In [9]:
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import gymnasium as gym
import numpy as np
import imageio
from PIL import Image, ImageDraw, ImageFont

# 비디오 프레임을 캡처하고 상단에 텍스트(예: 학습 스텝 수)를 그리는 헬퍼 함수
def record_episode(env_id, model=None, current_steps=0, max_steps=500):
    eval_env = gym.make(env_id, render_mode="rgb_array")
    obs, info = eval_env.reset(seed=42)
    frames = []
    done = False
    step_count = 0

    while not done and step_count < max_steps:
        # 프레임 가져오기
        frame = eval_env.render()

        # PIL을 사용해 이미지에 텍스트 그리기
        img = Image.fromarray(frame)
        draw = ImageDraw.Draw(img)
        text = f"Train Steps: {current_steps}"
        # 텍스트를 프레임 왼쪽 상단에 추가
        draw.text((10, 10), text, fill=(255, 0, 0))
        frames.append(np.array(img))

        if model is not None:
            action, _states = model.predict(obs, deterministic=True)
        else:
            action = eval_env.action_space.sample()

        obs, reward, terminated, truncated, info = eval_env.step(action)
        step_count += 1
        if terminated or truncated:
            done = True

    eval_env.close()
    return frames

In [10]:
# 1. 학습 전 무작위 에이전트 비디오 프레임 캡처
all_frames = []
print("학습 전 에피소드 녹화 중...")
all_frames.extend(record_episode("CartPole-v1", model=None, current_steps=0))

# 2. PPO 모델 초기화
model = PPO("MlpPolicy", "CartPole-v1", verbose=0, seed=42)

# 3. 단계별 학습 및 녹화 진행
target_steps = [1000, 3000, 5000] # 각 누적 스텝 시점
total_learned = 0

for steps in target_steps:
    learn_steps = steps - total_learned
    print(f"{steps} 스텝까지 학습 중...")
    model.learn(total_timesteps=learn_steps, reset_num_timesteps=False)
    total_learned = steps

    # 해당 시점의 플레이 녹화
    all_frames.extend(record_episode("CartPole-v1", model=model, current_steps=total_learned))

# 하나의 MP4 파일로 합쳐서 저장
video_path = "cartpole_training_progress.mp4"
imageio.mimsave(video_path, all_frames, fps=30)
print(f"비디오 저장 완료: {video_path}")

학습 전 에피소드 녹화 중...


/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


1000 스텝까지 학습 중...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


3000 스텝까지 학습 중...
5000 스텝까지 학습 중...


비디오 저장 완료: cartpole_training_progress.mp4


In [11]:
# 4. 저장한 비디오 재생
from IPython.display import Video
Video(video_path, embed=True, width=600, height=400)

In [12]:
# 5. 10개 에피소드 평균 보상 및 표준편차 평가
eval_env = gym.make("CartPole-v1")
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
eval_env.close()

print(f"[평가 결과] 10개 에피소드 평균 보상: {mean_reward} (표준편차: {std_reward})")

/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


[평가 결과] 10개 에피소드 평균 보상: 136.4 (표준편차: 28.841636569376572)
